# ANRF AISEHack Phase 2 Final Notebook

This notebook is a complete code-only pipeline for Phase 2. It:
- reads raw training data from Kaggle input paths,
- builds time-series train and validation windows (10 in, 16 out),
- trains a model from scratch,
- runs inference on test_in,
- writes /kaggle/working/preds.npy in the required format.

No external data, pretrained weights, or external artifacts are used.

## Strategy and Action Plan

### Strategy
1. Use only legal Phase 2 inputs: first 10 hours for all features.
2. Predict the next 16 hours of cpm25 over the 140 x 124 grid.
3. Optimize for both global and episode behavior using a composite objective:
   - L1 on normalized targets for stable training,
   - global SMAPE surrogate on denormalized predictions,
   - episode SMAPE surrogate using an episode mask,
   - episode correlation surrogate to align spatial episode patterns.
4. Build a fast episode mask proxy from climatology residuals:
   - anomaly = cpm25 - hourly climatology,
   - episodic if anomaly > 3 * sigma_grid and cpm25 > 1.
5. Use a multi-scale residual U-Net style network with coordinate channels.

### Action Plan
1. Discover months and usable features from raw data.
2. Build blocked train and validation splits from sliding windows.
3. Compute feature normalization statistics from competition data only.
4. Precompute episode proxy masks for loss weighting.
5. Train from scratch with mixed precision and best-checkpoint saving.
6. Infer on test_in and save strict /kaggle/working/preds.npy.

In [1]:
import gc

import random

from pathlib import Path



import numpy as np

import pandas as pd

import torch

import torch.nn as nn

import torch.nn.functional as F

from torch.utils.data import DataLoader, Dataset

from tqdm.auto import tqdm



SEED = 42



def seed_everything(seed=42):

    random.seed(seed)

    np.random.seed(seed)

    torch.manual_seed(seed)

    torch.cuda.manual_seed_all(seed)

    torch.backends.cudnn.deterministic = True

    torch.backends.cudnn.benchmark = False



seed_everything(SEED)



def select_runtime_device(min_supported_cc_major=7):

    if not torch.cuda.is_available():

        return torch.device("cpu"), False, "CUDA not available"



    try:

        cc_major, cc_minor = torch.cuda.get_device_capability(0)

        gpu_name = torch.cuda.get_device_name(0)

    except Exception:

        return torch.device("cpu"), False, "Unable to query CUDA capability"



    # Some Kaggle torch builds do not ship kernels for older GPU arch (e.g., sm60 P100).

    if cc_major < min_supported_cc_major:

        msg = (

            f"Detected GPU {gpu_name} with compute capability {cc_major}.{cc_minor}. "

            "This runtime can trigger 'no kernel image is available for execution on the device'. "

            "Switch accelerator to T4 x2 on Kaggle for stable CUDA execution. Falling back to CPU here."

        )

        return torch.device("cpu"), False, msg



    return torch.device("cuda"), True, f"Using GPU {gpu_name} (cc {cc_major}.{cc_minor})"



device, cuda_ok, device_message = select_runtime_device(min_supported_cc_major=7)

print(device_message)

print("Device:", device)



if device.type == "cuda":

    torch.backends.cuda.matmul.allow_tf32 = True

    torch.backends.cudnn.allow_tf32 = True



class CFG:

    comp_root = Path("/kaggle/input/competitions/anrf-aise-hack-phase-2-theme-2-pollution-forecasting-iitd/aisehack-theme-2")

    raw_root = comp_root / "raw"

    test_root = comp_root / "test_in"



    work_root = Path("/kaggle/working")

    model_path = work_root / "phase2_episode_best.pt"

    pred_path = work_root / "preds.npy"



    input_hours = 10

    output_hours = 16

    horizon = input_hours + output_hours



    val_ratio = 0.15

    min_gap_between_train_val = output_hours



    batch_size = 4

    epochs = 24

    lr = 2e-3

    weight_decay = 1e-4

    grad_clip = 1.0

    num_workers = 2

    use_amp = True

    base_channels = 48

    dropout = 0.05



    loss_w_l1 = 0.35

    loss_w_global_smape = 0.30

    loss_w_episode_smape = 0.20

    loss_w_episode_corr = 0.15



    stats_time_stride = 1



    features = [

        "cpm25",

        "q2",

        "t2",

        "u10",

        "v10",

        "swdown",

        "pblh",

        "psfc",

        "rain",

        "PM25",

        "NH3",

        "SO2",

        "NOx",

        "NMVOC_e",

        "NMVOC_finn",

        "bio",

    ]





CFG.work_root.mkdir(parents=True, exist_ok=True)


Using GPU Tesla T4 (cc 7.5)
Device: cuda


In [2]:
def discover_months(raw_root: Path):
    months = []
    for p in sorted(raw_root.iterdir()):
        if p.is_dir() and (p / "cpm25.npy").exists():
            months.append(p.name)
    return months

def to_hour_of_day(time_array):
    flat = np.asarray(time_array).reshape(-1)
    try:
        ts = pd.to_datetime(flat.astype(str), utc=True, errors="coerce")
        if np.all(pd.isna(ts)):
            raise ValueError("Unable to parse timestamps")
        return ts.hour.to_numpy(dtype=np.int16)
    except Exception:
        return (np.arange(len(flat)) % 24).astype(np.int16)

def load_memmaps(months, features):
    available = {}
    for month in months:
        month_dir = CFG.raw_root / month
        month_feats = {p.stem for p in month_dir.glob("*.npy")} - {"time", "lat_long"}
        available[month] = month_feats

    features_in_use = [f for f in features if all(f in available[m] for m in months)]
    if "cpm25" not in features_in_use:
        raise RuntimeError("cpm25 must be present in all months")

    raw_arrays = {}
    hours_of_day = {}

    for month in months:
        month_dir = CFG.raw_root / month
        raw_arrays[month] = {}
        for feat in features_in_use:
            raw_arrays[month][feat] = np.load(month_dir / f"{feat}.npy", mmap_mode="r")

        t_path = month_dir / "time.npy"
        if t_path.exists():
            hours_of_day[month] = to_hour_of_day(np.load(t_path, allow_pickle=True))
        else:
            t_len = raw_arrays[month]["cpm25"].shape[0]
            hours_of_day[month] = (np.arange(t_len) % 24).astype(np.int16)

    return raw_arrays, hours_of_day, features_in_use

def build_indices(months, raw_arrays, horizon, val_ratio, min_gap):
    train_idx = []
    val_idx = []
    info = {}

    for month in months:
        T = raw_arrays[month]["cpm25"].shape[0]
        n_windows = T - horizon + 1
        split = int((1.0 - val_ratio) * n_windows)
        train_end = max(split - min_gap, 1)

        month_train = [(month, s) for s in range(0, train_end)]
        month_val = [(month, s) for s in range(split, n_windows)]

        train_idx.extend(month_train)
        val_idx.extend(month_val)

        info[month] = {
            "hours": int(T),
            "windows": int(n_windows),
            "train_windows": int(len(month_train)),
            "val_windows": int(len(month_val)),
        }

    return train_idx, val_idx, info

assert CFG.raw_root.exists(), f"Missing raw path: {CFG.raw_root}"
assert CFG.test_root.exists(), f"Missing test path: {CFG.test_root}"

months = discover_months(CFG.raw_root)
print("Months found:", months)

raw_arrays, hours_of_day, features_in_use = load_memmaps(months, CFG.features)
print("Features used:", features_in_use)

H, W = raw_arrays[months[0]]["cpm25"].shape[1:]
print("Grid size:", (H, W))

train_index, val_index, split_info = build_indices(
    months, raw_arrays, CFG.horizon, CFG.val_ratio, CFG.min_gap_between_train_val
)

print("Train windows:", len(train_index))
print("Val windows:", len(val_index))
print("Split details:")
for m, d in split_info.items():
    print(m, d)

Months found: ['APRIL_16', 'DEC_16', 'JULY_16', 'OCT_16']
Features used: ['cpm25', 'q2', 't2', 'u10', 'v10', 'swdown', 'pblh', 'psfc', 'rain', 'PM25', 'NH3', 'SO2', 'NOx', 'NMVOC_e', 'NMVOC_finn', 'bio']
Grid size: (140, 124)
Train windows: 2340
Val windows: 428
Split details:
APRIL_16 {'hours': 715, 'windows': 690, 'train_windows': 570, 'val_windows': 104}
DEC_16 {'hours': 739, 'windows': 714, 'train_windows': 590, 'val_windows': 108}
JULY_16 {'hours': 739, 'windows': 714, 'train_windows': 590, 'val_windows': 108}
OCT_16 {'hours': 739, 'windows': 714, 'train_windows': 590, 'val_windows': 108}


In [3]:
def compute_feature_stats(raw_arrays, months, features, time_stride=1):
    stats = {}
    for feat in features:
        total_count = 0
        total_sum = 0.0
        total_sq = 0.0

        for month in months:
            arr = np.asarray(raw_arrays[month][feat][::time_stride], dtype=np.float64)
            total_count += arr.size
            total_sum += arr.sum()
            total_sq += (arr * arr).sum()

        mean = total_sum / max(total_count, 1)
        var = max((total_sq / max(total_count, 1)) - (mean * mean), 1e-8)
        stats[feat] = {"mean": float(mean), "std": float(np.sqrt(var))}

    return stats

def compute_hourly_climatology(raw_arrays, months, hours_of_day):
    csum = None
    ccnt = np.zeros((24, 1, 1), dtype=np.int64)

    for month in months:
        cpm = np.asarray(raw_arrays[month]["cpm25"], dtype=np.float32)
        if csum is None:
            csum = np.zeros((24, cpm.shape[1], cpm.shape[2]), dtype=np.float64)

        hrs = hours_of_day[month]
        for h in range(24):
            mask = hrs == h
            if mask.any():
                csum[h] += cpm[mask].sum(axis=0, dtype=np.float64)
                ccnt[h] += int(mask.sum())

    clim = csum / np.maximum(ccnt, 1)
    return clim.astype(np.float32)

def compute_episode_sigma(raw_arrays, months, hours_of_day, climatology):
    sq_sum = np.zeros_like(climatology[0], dtype=np.float64)
    count = 0

    for month in months:
        cpm = np.asarray(raw_arrays[month]["cpm25"], dtype=np.float32)
        hrs = hours_of_day[month]
        anomaly = cpm - climatology[hrs]
        sq_sum += (anomaly.astype(np.float64) ** 2).sum(axis=0)
        count += anomaly.shape[0]

    sigma = np.sqrt(sq_sum / max(count, 1)).astype(np.float32)
    sigma = np.maximum(sigma, 1e-3)
    return sigma

def precompute_episode_masks(raw_arrays, months, hours_of_day, climatology, sigma):
    masks = {}
    threshold = 3.0 * sigma[None, :, :]

    for month in months:
        cpm = np.asarray(raw_arrays[month]["cpm25"], dtype=np.float32)
        hrs = hours_of_day[month]
        anomaly = cpm - climatology[hrs]
        mask = ((anomaly > threshold) & (cpm > 1.0)).astype(np.float32)
        masks[month] = mask

    return masks

feature_stats = compute_feature_stats(
    raw_arrays, months, features_in_use, time_stride=CFG.stats_time_stride
)

print("Feature stats (mean/std):")
for f in features_in_use:
    print(f, feature_stats[f])

cpm_mean = feature_stats["cpm25"]["mean"]
cpm_std = feature_stats["cpm25"]["std"]

climatology = compute_hourly_climatology(raw_arrays, months, hours_of_day)
episode_sigma = compute_episode_sigma(raw_arrays, months, hours_of_day, climatology)
episode_masks = precompute_episode_masks(raw_arrays, months, hours_of_day, climatology, episode_sigma)

for month in months:
    print(month, "episode ratio:", float(episode_masks[month].mean()))

gc.collect()

Feature stats (mean/std):
cpm25 {'mean': 33.60772606860789, 'std': 52.27758568185381}
q2 {'mean': 0.011485480463106818, 'std': 0.007025644124585724}
t2 {'mean': 291.5940407566613, 'std': 14.066832541565434}
u10 {'mean': 1.5475738702426902, 'std': 3.5471407293487083}
v10 {'mean': 0.1373008272251088, 'std': 2.9958337250435743}
swdown {'mean': 221.78057793182003, 'std': 309.4067190565461}
pblh {'mean': 764.290133411197, 'std': 636.3435981957224}
psfc {'mean': 87987.0068799978, 'std': 17645.765546759645}
rain {'mean': 0.09479472458538274, 'std': 1.070709312144718}
PM25 {'mean': 3.4916783287204745e-11, 'std': 0.0001}
NH3 {'mean': 3.0148893268202665e-11, 'std': 0.0001}
SO2 {'mean': 3.8192051191775635e-11, 'std': 0.0001}
NOx {'mean': 4.129316599389643e-11, 'std': 0.0001}
NMVOC_e {'mean': 4.85842857157431e-11, 'std': 0.0001}
NMVOC_finn {'mean': 4.1703270872943727e-10, 'std': 0.0001}
bio {'mean': 5.1812730451369e-11, 'std': 0.0001}
APRIL_16 episode ratio: 0.011991556733846664
DEC_16 episode rat

41

In [4]:
class PM25EpisodeDataset(Dataset):
    def __init__(
        self,
        raw_arrays,
        feature_stats,
        episode_masks,
        features,
        index_list,
        input_hours=10,
        output_hours=16,
    ):
        self.raw_arrays = raw_arrays
        self.feature_stats = feature_stats
        self.episode_masks = episode_masks
        self.features = features
        self.index_list = index_list
        self.input_hours = input_hours
        self.output_hours = output_hours
        self.horizon = input_hours + output_hours

        example_month = index_list[0][0]
        self.H = raw_arrays[example_month]["cpm25"].shape[1]
        self.W = raw_arrays[example_month]["cpm25"].shape[2]

    def __len__(self):
        return len(self.index_list)

    def _norm(self, arr, feat):
        st = self.feature_stats[feat]
        return (arr - st["mean"]) / st["std"]

    def __getitem__(self, idx):
        month, start = self.index_list[idx]
        in_end = start + self.input_hours
        out_end = start + self.horizon

        x = np.empty((self.input_hours, self.H, self.W, len(self.features)), dtype=np.float32)

        for c, feat in enumerate(self.features):
            arr = self.raw_arrays[month][feat][start:in_end].astype(np.float32)
            x[..., c] = self._norm(arr, feat)

        y_raw = self.raw_arrays[month]["cpm25"][in_end:out_end].astype(np.float32)
        y = self._norm(y_raw, "cpm25")

        m = self.episode_masks[month][in_end:out_end].astype(np.float32)

        return torch.from_numpy(x), torch.from_numpy(y), torch.from_numpy(m)

pin_memory = device.type == "cuda"

train_ds = PM25EpisodeDataset(
    raw_arrays=raw_arrays,
    feature_stats=feature_stats,
    episode_masks=episode_masks,
    features=features_in_use,
    index_list=train_index,
    input_hours=CFG.input_hours,
    output_hours=CFG.output_hours,
)

val_ds = PM25EpisodeDataset(
    raw_arrays=raw_arrays,
    feature_stats=feature_stats,
    episode_masks=episode_masks,
    features=features_in_use,
    index_list=val_index,
    input_hours=CFG.input_hours,
    output_hours=CFG.output_hours,
)

train_loader = DataLoader(
    train_ds,
    batch_size=CFG.batch_size,
    shuffle=True,
    num_workers=CFG.num_workers,
    pin_memory=pin_memory,
    drop_last=True,
    persistent_workers=(CFG.num_workers > 0),
)

val_loader = DataLoader(
    val_ds,
    batch_size=CFG.batch_size,
    shuffle=False,
    num_workers=CFG.num_workers,
    pin_memory=pin_memory,
    drop_last=False,
    persistent_workers=(CFG.num_workers > 0),
)

xb, yb, mb = next(iter(train_loader))
print("Batch shapes -> x:", tuple(xb.shape), "y:", tuple(yb.shape), "episode_mask:", tuple(mb.shape))

Batch shapes -> x: (4, 10, 140, 124, 16) y: (4, 16, 140, 124) episode_mask: (4, 16, 140, 124)


In [5]:
def make_group_norm(channels):
    for g in [16, 8, 4, 2, 1]:
        if channels % g == 0:
            return nn.GroupNorm(g, channels)
    return nn.GroupNorm(1, channels)

class ResBlock(nn.Module):
    def __init__(self, in_ch, out_ch, dropout=0.0):
        super().__init__()
        self.conv1 = nn.Conv2d(in_ch, out_ch, kernel_size=3, padding=1, bias=False)
        self.norm1 = make_group_norm(out_ch)
        self.conv2 = nn.Conv2d(out_ch, out_ch, kernel_size=3, padding=1, bias=False)
        self.norm2 = make_group_norm(out_ch)
        self.act = nn.SiLU(inplace=True)
        self.drop = nn.Dropout2d(dropout) if dropout > 0 else nn.Identity()
        self.skip = nn.Conv2d(in_ch, out_ch, kernel_size=1, bias=False) if in_ch != out_ch else nn.Identity()

    def forward(self, x):
        residual = self.skip(x)
        x = self.act(self.norm1(self.conv1(x)))
        x = self.drop(x)
        x = self.norm2(self.conv2(x))
        x = self.act(x + residual)
        return x

class DownBlock(nn.Module):
    def __init__(self, in_ch, out_ch, dropout=0.0):
        super().__init__()
        self.pool = nn.MaxPool2d(2)
        self.block = ResBlock(in_ch, out_ch, dropout=dropout)

    def forward(self, x):
        x = self.pool(x)
        x = self.block(x)
        return x

class UpBlock(nn.Module):
    def __init__(self, in_ch, skip_ch, out_ch, dropout=0.0):
        super().__init__()
        self.up = nn.ConvTranspose2d(in_ch, out_ch, kernel_size=2, stride=2)
        self.block = ResBlock(out_ch + skip_ch, out_ch, dropout=dropout)

    def forward(self, x, skip):
        x = self.up(x)

        dy = skip.size(2) - x.size(2)
        dx = skip.size(3) - x.size(3)
        if dy != 0 or dx != 0:
            x = F.pad(x, [dx // 2, dx - dx // 2, dy // 2, dy - dy // 2])

        x = torch.cat([x, skip], dim=1)
        x = self.block(x)
        return x

class EpisodeUNet(nn.Module):
    def __init__(self, n_features, input_hours=10, output_hours=16, base_ch=48, dropout=0.05):
        super().__init__()
        in_ch = input_hours * n_features + 2

        self.enc1 = ResBlock(in_ch, base_ch, dropout=dropout)
        self.enc2 = DownBlock(base_ch, base_ch * 2, dropout=dropout)
        self.enc3 = DownBlock(base_ch * 2, base_ch * 4, dropout=dropout)

        self.bottleneck = ResBlock(base_ch * 4, base_ch * 4, dropout=dropout)

        self.up2 = UpBlock(base_ch * 4, base_ch * 2, base_ch * 2, dropout=dropout)
        self.up1 = UpBlock(base_ch * 2, base_ch, base_ch, dropout=dropout)

        self.head = nn.Conv2d(base_ch, output_hours, kernel_size=1)

    @staticmethod
    def get_grid(batch, h, w, device):
        yy = torch.linspace(-1.0, 1.0, h, device=device).view(1, 1, h, 1).expand(batch, 1, h, w)
        xx = torch.linspace(-1.0, 1.0, w, device=device).view(1, 1, 1, w).expand(batch, 1, h, w)
        return torch.cat([yy, xx], dim=1)

    def forward(self, x):
        # x: (B, T, H, W, F)
        B, T, Hh, Ww, Ff = x.shape
        x = x.permute(0, 1, 4, 2, 3).reshape(B, T * Ff, Hh, Ww)
        grid = self.get_grid(B, Hh, Ww, x.device)
        x = torch.cat([x, grid], dim=1)

        s1 = self.enc1(x)
        s2 = self.enc2(s1)
        s3 = self.enc3(s2)

        b = self.bottleneck(s3)

        d2 = self.up2(b, s2)
        d1 = self.up1(d2, s1)

        out = self.head(d1)
        return out

model = EpisodeUNet(
    n_features=len(features_in_use),
    input_hours=CFG.input_hours,
    output_hours=CFG.output_hours,
    base_ch=CFG.base_channels,
    dropout=CFG.dropout,
).to(device)

n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print("Trainable parameters:", f"{n_params:,}")

Trainable parameters: 1,837,024


In [6]:
def smape_map(pred, target, eps=1e-3):

    return torch.abs(pred - target) / (0.5 * (torch.abs(pred) + torch.abs(target)) + eps)



def masked_mean(values, mask, eps=1e-6):
    return (values * mask).sum() / (mask.sum() + eps)



def episode_corr_loss(pred, target, mask, eps=1e-6):

    # pred/target/mask: (B, T, H, W)
    B, T, _, _ = pred.shape
    p = pred.view(B, T, -1)
    y = target.view(B, T, -1)
    m = mask.view(B, T, -1)

    losses = []
    for t in range(T):
        pt = p[:, t, :]
        yt = y[:, t, :]
        mt = m[:, t, :]

        wsum = mt.sum(dim=1, keepdim=True)
        valid = (wsum.squeeze(1) > 50).float()

        pm = (pt * mt).sum(dim=1, keepdim=True) / (wsum + eps)
        ym = (yt * mt).sum(dim=1, keepdim=True) / (wsum + eps)

        pc = (pt - pm) * mt
        yc = (yt - ym) * mt

        num = (pc * yc).sum(dim=1)
        den = torch.sqrt((pc.pow(2).sum(dim=1) * yc.pow(2).sum(dim=1)) + eps)
        corr_masked = num / (den + eps)

        p_all = pt - pt.mean(dim=1, keepdim=True)
        y_all = yt - yt.mean(dim=1, keepdim=True)
        num_all = (p_all * y_all).sum(dim=1)
        den_all = torch.sqrt((p_all.pow(2).sum(dim=1) * y_all.pow(2).sum(dim=1)) + eps)
        corr_global = num_all / (den_all + eps)

        corr = valid * corr_masked + (1.0 - valid) * corr_global
        losses.append(1.0 - corr.mean())

    return torch.stack(losses).mean()



def compute_losses(pred_norm, target_norm, episode_mask, cpm_mean, cpm_std):
    # Stable pointwise term in normalized space
    l1 = F.l1_loss(pred_norm, target_norm)

    # Metric-aligned terms in physical space
    pred = pred_norm * cpm_std + cpm_mean
    target = target_norm * cpm_std + cpm_mean

    s_map = smape_map(pred, target)
    global_smape = s_map.mean()
    episode_smape = masked_mean(s_map, episode_mask)
    episode_corr = episode_corr_loss(pred, target, episode_mask)

    total = (
        CFG.loss_w_l1 * l1
        + CFG.loss_w_global_smape * global_smape
        + CFG.loss_w_episode_smape * episode_smape
        + CFG.loss_w_episode_corr * episode_corr
    )

    parts = {
        "loss": float(total.detach().cpu()),
        "l1": float(l1.detach().cpu()),
        "global_smape": float(global_smape.detach().cpu()),
        "episode_smape": float(episode_smape.detach().cpu()),
        "episode_corr_loss": float(episode_corr.detach().cpu()),
    }

    return total, parts

@torch.no_grad()
def evaluate(model, loader, cpm_mean, cpm_std):
    model.eval()
    totals = {
        "loss": 0.0,
        "l1": 0.0,
        "global_smape": 0.0,
        "episode_smape": 0.0,
        "episode_corr_loss": 0.0,
    }

    n = 0
    for xb, yb, mb in loader:
        xb = xb.to(device, non_blocking=True)
        yb = yb.to(device, non_blocking=True)
        mb = mb.to(device, non_blocking=True)

        pred = model(xb)
        parts = compute_losses(pred, yb, mb, cpm_mean, cpm_std)[1]

        bs = xb.size(0)
        n += bs
        for k in totals:
            totals[k] += parts[k] * bs

    for k in totals:
        totals[k] /= max(n, 1)

    return totals


In [7]:
optimizer = torch.optim.AdamW(model.parameters(), lr=CFG.lr, weight_decay=CFG.weight_decay)



scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=CFG.epochs, eta_min=CFG.lr * 0.05)



amp_enabled = CFG.use_amp and device.type == "cuda" and cuda_ok

autocast_device = "cuda" if device.type == "cuda" else "cpu"

scaler = torch.amp.GradScaler("cuda", enabled=amp_enabled)



best_val = float("inf")

best_epoch = -1

history = []



patience = 6

patience_counter = 0



print("Starting training...")



for epoch in range(1, CFG.epochs + 1):

    model.train()



    running = {

        "loss": 0.0,

        "l1": 0.0,

        "global_smape": 0.0,

        "episode_smape": 0.0,

        "episode_corr_loss": 0.0,

    }



    n_seen = 0

    pbar = tqdm(train_loader, desc=f"Epoch {epoch:02d}/{CFG.epochs}")

    for xb, yb, mb in pbar:

        xb = xb.to(device, non_blocking=True)

        yb = yb.to(device, non_blocking=True)

        mb = mb.to(device, non_blocking=True)



        optimizer.zero_grad(set_to_none=True)



        with torch.amp.autocast(autocast_device, enabled=amp_enabled):

            pred = model(xb)

            loss, parts = compute_losses(pred, yb, mb, cpm_mean, cpm_std)



        scaler.scale(loss).backward()

        scaler.unscale_(optimizer)



        nn.utils.clip_grad_norm_(model.parameters(), CFG.grad_clip)

        scaler.step(optimizer)

        scaler.update()



        bs = xb.size(0)

        n_seen += bs

        for k in running:

            running[k] += parts[k] * bs



        pbar.set_postfix({

            "loss": f"{running['loss']/max(n_seen,1):.4f}",

            "g_smape": f"{running['global_smape']/max(n_seen,1):.4f}",

            "ep_smape": f"{running['episode_smape']/max(n_seen,1):.4f}",

        })



    scheduler.step()



    train_metrics = {k: running[k] / max(n_seen, 1) for k in running}

    val_metrics = evaluate(model, val_loader, cpm_mean, cpm_std)



    row = {

        "epoch": epoch,

        "lr": float(optimizer.param_groups[0]["lr"]),

        "train": train_metrics,

        "val": val_metrics,

    }



    history.append(row)



    print(

        f"Epoch {epoch:02d} | "

        f"train_loss={train_metrics['loss']:.5f} | "

        f"val_loss={val_metrics['loss']:.5f} | "

        f"val_gSMAPE={val_metrics['global_smape']:.5f} | "

        f"val_epSMAPE={val_metrics['episode_smape']:.5f} | "

        f"val_epCorrLoss={val_metrics['episode_corr_loss']:.5f}"

    )



    if val_metrics["loss"] < best_val:

        best_val = val_metrics["loss"]

        best_epoch = epoch

        patience_counter = 0



        torch.save(

            {

                "model_state_dict": model.state_dict(),

                "feature_stats": feature_stats,

                "features_in_use": features_in_use,

                "best_val_loss": best_val,

                "best_epoch": best_epoch,

                "history": history,

                "cpm_mean": cpm_mean,

                "cpm_std": cpm_std,

                "model_config": {

                    "base_channels": CFG.base_channels,

                    "dropout": CFG.dropout,

                    "input_hours": CFG.input_hours,

                    "output_hours": CFG.output_hours,

                    "num_features": len(features_in_use),

                },

            },



            CFG.model_path,

        )



        print(f"[BEST] Saved checkpoint to {CFG.model_path} at epoch {epoch}")



    else:

        patience_counter += 1



    if patience_counter >= patience:

        print(f"Early stopping triggered after {patience} epochs without improvement.")

        break



print("Training finished. Best epoch:", best_epoch, "Best val loss:", best_val)



if device.type == "cpu" and not cuda_ok:

    print("Note: Running on CPU fallback due to unsupported CUDA architecture. Switch to T4 x2 for GPU training.")


Starting training...


Epoch 01/24:   0%|          | 0/585 [00:00<?, ?it/s]

Epoch 01 | train_loss=0.28830 | val_loss=0.26547 | val_gSMAPE=0.39336 | val_epSMAPE=0.27869 | val_epCorrLoss=0.16684
[BEST] Saved checkpoint to /kaggle/working/phase2_episode_best.pt at epoch 1


Epoch 02/24:   0%|          | 0/585 [00:00<?, ?it/s]

Epoch 02 | train_loss=0.22960 | val_loss=0.24107 | val_gSMAPE=0.32392 | val_epSMAPE=0.29905 | val_epCorrLoss=0.14910
[BEST] Saved checkpoint to /kaggle/working/phase2_episode_best.pt at epoch 2


...

Epoch 22 | train_loss=0.09355 | val_loss=0.17334 | val_gSMAPE=0.21584 | val_epSMAPE=0.24402 | val_epCorrLoss=0.12246


Epoch 23/24:   0%|          | 0/585 [00:00<?, ?it/s]

Epoch 23 | train_loss=0.09264 | val_loss=0.17397 | val_gSMAPE=0.21564 | val_epSMAPE=0.24525 | val_epCorrLoss=0.12428


Epoch 24/24:   0%|          | 0/585 [00:00<?, ?it/s]

Epoch 24 | train_loss=0.09206 | val_loss=0.17416 | val_gSMAPE=0.21645 | val_epSMAPE=0.24509 | val_epCorrLoss=0.12442
Training finished. Best epoch: 21 Best val loss: 0.1727148325086754


In [8]:
if CFG.model_path.exists():

    checkpoint = torch.load(CFG.model_path, map_location=device)

    model.load_state_dict(checkpoint["model_state_dict"])

    print("Loaded best checkpoint from:", CFG.model_path)

    print("Best epoch:", checkpoint.get("best_epoch"), "Best val loss:", checkpoint.get("best_val_loss"))



else:

    raise FileNotFoundError(f"Checkpoint not found: {CFG.model_path}")



model.eval()

amp_enabled = CFG.use_amp and device.type == "cuda" and cuda_ok

autocast_device = "cuda" if device.type == "cuda" else "cpu"



test_arrays = {}

for feat in features_in_use:

    fpath = CFG.test_root / f"{feat}.npy"

    if not fpath.exists():

        raise FileNotFoundError(f"Missing test feature: {fpath}")



    test_arrays[feat] = np.load(fpath, mmap_mode="r")



n_test = test_arrays["cpm25"].shape[0]

test_timesteps = test_arrays["cpm25"].shape[1]

print("Test samples:", n_test)

print("Test timesteps provided:", test_timesteps)



if test_timesteps < CFG.input_hours:

    raise ValueError("Test data has fewer than 10 timesteps")



if n_test != 218:

    print("Warning: competition guide indicates 218 test samples, but found", n_test)



pred_norm = np.zeros((n_test, CFG.output_hours, H, W), dtype=np.float32)

infer_bs = max(1, CFG.batch_size * 2)



with torch.no_grad():

    for s in tqdm(range(0, n_test, infer_bs), desc="Inference"):

        e = min(s + infer_bs, n_test)

        x = np.empty((e - s, CFG.input_hours, H, W, len(features_in_use)), dtype=np.float32)



        for c, feat in enumerate(features_in_use):

            arr = test_arrays[feat][s:e, :CFG.input_hours].astype(np.float32)

            st = feature_stats[feat]

            x[..., c] = (arr - st["mean"]) / st["std"]



        xb = torch.from_numpy(x).to(device, non_blocking=True)



        with torch.amp.autocast(autocast_device, enabled=amp_enabled):

            out = model(xb)



        pred_norm[s:e] = out.detach().cpu().numpy().astype(np.float32)



pred = pred_norm * cpm_std + cpm_mean

pred = np.clip(pred, 0.0, None)

pred = np.transpose(pred, (0, 2, 3, 1)).astype(np.float32)



np.save(CFG.pred_path, pred)

print("Saved predictions to:", CFG.pred_path)

print("Prediction shape:", pred.shape)

print("Min/Max:", float(pred.min()), float(pred.max()))

print("Has NaN:", bool(np.isnan(pred).any()), "Has Inf:", bool(np.isinf(pred).any()))


Loaded best checkpoint from: /kaggle/working/phase2_episode_best.pt
Best epoch: 21 Best val loss: 0.1727148325086754
Test samples: 218
Test timesteps provided: 10


Inference:   0%|          | 0/28 [00:00<?, ?it/s]

Saved predictions to: /kaggle/working/preds.npy
Prediction shape: (218, 140, 124, 16)
Min/Max: 0.0 1485.9443359375
Has NaN: False Has Inf: False


In [9]:
preds = np.load(CFG.pred_path)

print("Final submission checks")
print("- File exists:", CFG.pred_path.exists())
print("- Shape:", preds.shape)
print("- Last dims are (140, 124, 16):", preds.shape[1:] == (140, 124, 16))
print("- Dtype is float32/float64:", preds.dtype in (np.float32, np.float64))
print("- No NaN:", not np.isnan(preds).any())
print("- No Inf:", not np.isinf(preds).any())

if preds.shape[1:] != (140, 124, 16):
    raise ValueError(f"Invalid spatial/temporal shape: {preds.shape}")

if preds.shape[0] != 218:
    print("Warning: expected first dimension 218 as per competition guide.")

print("preds.npy is ready at /kaggle/working/preds.npy")

Final submission checks
- File exists: True
- Shape: (218, 140, 124, 16)
- Last dims are (140, 124, 16): True
- Dtype is float32/float64: True
- No NaN: True
- No Inf: True
preds.npy is ready at /kaggle/working/preds.npy


In [11]:
# Closest-possible offline scorer using published competition formulas on seen windows.
# Important: this is still NOT the official Kaggle leaderboard score.
# Reason: hidden 2017 labels are unavailable and final metric weights are undisclosed.

try:
    from statsmodels.tsa.seasonal import STL
except Exception as e:
    raise ImportError("statsmodels is required for the published STL episode method.") from e

n_eval_windows = min(128, len(train_index))
print(f"Evaluating published metric components on {n_eval_windows} seen training windows...")

if not CFG.model_path.exists():
    raise FileNotFoundError(f"Model checkpoint not found: {CFG.model_path}")

ckpt = torch.load(CFG.model_path, map_location=device)
model.load_state_dict(ckpt["model_state_dict"])
model.eval()

amp_enabled_eval = bool(device.type == "cuda" and ("cuda_ok" in globals()) and cuda_ok)
autocast_device = "cuda" if device.type == "cuda" else "cpu"

eps = 1e-8
rng = np.random.default_rng(SEED)
chosen_idx = rng.choice(len(train_index), size=n_eval_windows, replace=False)
chosen_windows = [train_index[int(i)] for i in chosen_idx]
chosen_months = sorted({m for m, _ in chosen_windows})
print("Months covered in sampled windows:", chosen_months)


def _mask_cache_path(month_name):
    return CFG.work_root / f"official_episode_mask_{month_name}.npy"


def _validate_single_mask(mask_arr, month_name):
    arr = np.asarray(mask_arr)
    ref_shape = raw_arrays[month_name]["cpm25"].shape

    if arr.shape != ref_shape:
        return False, f"shape mismatch {arr.shape} vs {ref_shape}"
    if arr.ndim != 3:
        return False, f"expected 3D mask, got ndim={arr.ndim}"
    if not np.issubdtype(arr.dtype, np.number):
        return False, f"non-numeric dtype {arr.dtype}"

    # Fast binary check on a stride sample.
    flat = arr.reshape(-1)
    stride = max(1, flat.size // 200000)
    sample = flat[::stride]
    if not np.isin(sample, [0, 1]).all():
        return False, "mask values are not binary in sampled check"

    return True, "ok"


def compute_official_episode_mask_for_month(month_name):
    """
    Published helper-notebook episode logic:
    - STL(ts, period=24, robust=True)
    - episodic if remainder > 3 * std(remainder) and cpm25 > 1
    """
    cpm = np.asarray(raw_arrays[month_name]["cpm25"], dtype=np.float32)
    Tm, Hm, Wm = cpm.shape
    cpm_2d = cpm.reshape(Tm, -1)

    mask_2d = np.zeros_like(cpm_2d, dtype=np.uint8)

    for k in tqdm(range(cpm_2d.shape[1]), desc=f"STL episode mask {month_name}", leave=False):
        ts = cpm_2d[:, k]
        stl = STL(ts, period=24, robust=True)
        res = stl.fit()
        remainder = res.resid

        sigma = np.std(remainder) + eps
        mask_2d[:, k] = ((remainder > 3.0 * sigma) & (ts > 1.0)).astype(np.uint8)

    return mask_2d.reshape(Tm, Hm, Wm)


# -------------------- Reuse mask if already computed exactly --------------------
official_episode_masks_local = {}

# 1) Try memory reuse from previous runs of this exact scorer cell.
existing_official_masks = globals().get("official_episode_masks", None)
if isinstance(existing_official_masks, dict):
    for month in chosen_months:
        if month in existing_official_masks:
            ok, msg = _validate_single_mask(existing_official_masks[month], month)
            if ok:
                official_episode_masks_local[month] = np.asarray(existing_official_masks[month], dtype=np.uint8)
            else:
                print(f"Memory mask for {month} not reusable: {msg}")

if official_episode_masks_local:
    print(f"Reused {len(official_episode_masks_local)} month mask(s) from memory.")

# 2) Try disk cache reuse for remaining months.
for month in chosen_months:
    if month in official_episode_masks_local:
        continue

    cache_path = _mask_cache_path(month)
    if cache_path.exists():
        cached = np.load(cache_path, mmap_mode="r")
        ok, msg = _validate_single_mask(cached, month)
        if ok:
            official_episode_masks_local[month] = np.asarray(cached, dtype=np.uint8)
            print(f"Loaded cached official mask for {month} from {cache_path}")
        else:
            print(f"Cached mask for {month} not reusable: {msg}")

# 3) If still missing, compute only missing months with STL.
missing_months = [m for m in chosen_months if m not in official_episode_masks_local]
if missing_months:
    if "episode_masks" in globals():
        print("Found episode_masks from earlier cells, but not using them here because this scorer requires STL-based official masks.")

    print("Computing STL official masks for missing months:", missing_months)
    for month in missing_months:
        mask_month = compute_official_episode_mask_for_month(month)
        official_episode_masks_local[month] = mask_month.astype(np.uint8)

        # Save cache for faster reruns.
        cache_path = _mask_cache_path(month)
        np.save(cache_path, official_episode_masks_local[month])
        print(f"Saved official mask cache: {cache_path}")
else:
    print("All required official masks were reused; no STL recomputation needed.")

# Expose for future reruns in same session.
official_episode_masks = official_episode_masks_local

# -------------------- Score computation --------------------
# Metric accumulators (time-step level as in published formulas)
global_smape_t_all = []
episode_smape_t_all = []
episode_corr_t_all = []
no_episode_t_count = 0

eval_batch_size = 16

with torch.no_grad():
    for s in tqdm(range(0, len(chosen_windows), eval_batch_size), desc="Scoring windows"):
        batch_windows = chosen_windows[s:s + eval_batch_size]
        B = len(batch_windows)

        x_batch = np.empty((B, CFG.input_hours, H, W, len(features_in_use)), dtype=np.float32)
        y_batch = np.empty((B, CFG.output_hours, H, W), dtype=np.float32)
        month_list = []
        in_end_list = []

        for b, (month, start) in enumerate(batch_windows):
            in_end = start + CFG.input_hours
            out_end = start + CFG.horizon

            for c, feat in enumerate(features_in_use):
                arr = raw_arrays[month][feat][start:in_end].astype(np.float32)
                st = feature_stats[feat]
                x_batch[b, ..., c] = (arr - st["mean"]) / st["std"]

            y_batch[b] = raw_arrays[month]["cpm25"][in_end:out_end].astype(np.float32)
            month_list.append(month)
            in_end_list.append(in_end)

        xb = torch.from_numpy(x_batch).to(device, non_blocking=True)
        with torch.amp.autocast(autocast_device, enabled=amp_enabled_eval):
            pred_norm = model(xb)

        pred_batch = pred_norm.detach().cpu().numpy().astype(np.float32)
        pred_batch = pred_batch * cpm_std + cpm_mean
        pred_batch = np.clip(pred_batch, 0.0, None)

        for b in range(B):
            month = month_list[b]
            in_end = in_end_list[b]
            y_true = y_batch[b]
            pred = pred_batch[b]

            for t in range(CFG.output_hours):
                gt_t = y_true[t]
                pr_t = pred[t]

                # 1) Global SMAPE_t (all grids)
                denom_g = 0.5 * (np.abs(gt_t) + np.abs(pr_t)) + eps
                smape_t_global = np.mean(np.abs(pr_t - gt_t) / denom_g)
                global_smape_t_all.append(float(smape_t_global))

                # Episode set E_t from official STL mask
                abs_t = in_end + t
                et_mask = official_episode_masks[month][abs_t] > 0

                if et_mask.sum() == 0:
                    no_episode_t_count += 1
                    continue

                # 2) Episode SMAPE_t (episodic grids only)
                gt_ep = gt_t[et_mask]
                pr_ep = pr_t[et_mask]
                denom_e = 0.5 * (np.abs(gt_ep) + np.abs(pr_ep)) + eps
                smape_t_episode = np.mean(np.abs(pr_ep - gt_ep) / denom_e)
                episode_smape_t_all.append(float(smape_t_episode))

                # 3) Episode Corr_t (episodic grids only)
                if gt_ep.size > 1 and np.std(gt_ep) > eps and np.std(pr_ep) > eps:
                    corr_t = np.corrcoef(gt_ep, pr_ep)[0, 1]
                else:
                    corr_t = 0.0

                if not np.isfinite(corr_t):
                    corr_t = 0.0

                episode_corr_t_all.append(float(corr_t))


# Published aggregate component metrics
GlobalSMAPE = float(np.mean(global_smape_t_all)) if global_smape_t_all else np.nan
EpisodeSMAPE = float(np.mean(episode_smape_t_all)) if episode_smape_t_all else np.nan
EpisodeCorr = float(np.mean(episode_corr_t_all)) if episode_corr_t_all else np.nan

# Published normalization
NormEpisodeCorr = (EpisodeCorr + 1.0) / 2.0 if np.isfinite(EpisodeCorr) else np.nan
NormGlobalSMAPE = 1.0 - (GlobalSMAPE / 2.0) if np.isfinite(GlobalSMAPE) else np.nan
NormEpisodeSMAPE = 1.0 - (EpisodeSMAPE / 2.0) if np.isfinite(EpisodeSMAPE) else np.nan

# Final weights are undisclosed; only report plausible band and equal-weight proxy
components = np.array([NormGlobalSMAPE, NormEpisodeCorr, NormEpisodeSMAPE], dtype=np.float64)
if np.all(np.isfinite(components)):
    proxy_equal_weight = float(np.mean(components))
    possible_min = float(np.min(components))
    possible_max = float(np.max(components))
else:
    proxy_equal_weight = np.nan
    possible_min = np.nan
    possible_max = np.nan

print("\nPublished component metrics on seen windows")
print(f"GlobalSMAPE  : {GlobalSMAPE:.6f}")
print(f"EpisodeSMAPE : {EpisodeSMAPE:.6f}")
print(f"EpisodeCorr  : {EpisodeCorr:.6f}")

print("\nPublished normalized components")
print(f"NormGlobalSMAPE : {NormGlobalSMAPE:.6f}")
print(f"NormEpisodeSMAPE: {NormEpisodeSMAPE:.6f}")
print(f"NormEpisodeCorr : {NormEpisodeCorr:.6f}")

print("\nUnknown final weights => no exact offline final score")
print(f"Equal-weight proxy           : {proxy_equal_weight:.6f}")
print(f"Plausible weighted-score band: [{possible_min:.6f}, {possible_max:.6f}]")

print("\nNotes")
print(f"- Timesteps with no episode points in sampled windows: {no_episode_t_count}")
print("- Evaluated on seen training windows, so this is optimistic.")
print("- Official leaderboard score requires hidden 2017 labels + undisclosed weights.")


Evaluating published metric components on 128 seen training windows...
Months covered in sampled windows: ['APRIL_16', 'DEC_16', 'JULY_16', 'OCT_16']
Found episode_masks from earlier cells, but not using them here because this scorer requires STL-based official masks.
Computing STL official masks for missing months: ['APRIL_16', 'DEC_16', 'JULY_16', 'OCT_16']


STL episode mask APRIL_16:   0%|          | 0/17360 [00:00<?, ?it/s]

Saved official mask cache: /kaggle/working/official_episode_mask_APRIL_16.npy


STL episode mask DEC_16:   0%|          | 0/17360 [00:00<?, ?it/s]

Saved official mask cache: /kaggle/working/official_episode_mask_DEC_16.npy


...

Saved official mask cache: /kaggle/working/official_episode_mask_JULY_16.npy


STL episode mask OCT_16:   0%|          | 0/17360 [00:00<?, ?it/s]

Saved official mask cache: /kaggle/working/official_episode_mask_OCT_16.npy


Scoring windows:   0%|          | 0/8 [00:00<?, ?it/s]


Published component metrics on seen windows
GlobalSMAPE  : 0.152841
EpisodeSMAPE : 0.183923
EpisodeCorr  : 0.964089

Published normalized components
NormGlobalSMAPE : 0.923580
NormEpisodeSMAPE: 0.908039
NormEpisodeCorr : 0.982044

Unknown final weights => no exact offline final score
Equal-weight proxy           : 0.937887
Plausible weighted-score band: [0.908039, 0.982044]

Notes
- Timesteps with no episode points in sampled windows: 0
- Evaluated on seen training windows, so this is optimistic.
- Official leaderboard score requires hidden 2017 labels + undisclosed weights.
